In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_percentage_error

from loaders._load_vn30_reg_deep import preprocess, VN30, TARGETS
from models.regression.tft import TemporalFusionTransformer

In [3]:
train_loader, valid_loader, test_loader, scaler = preprocess('ACB', 'tft', verbose=True)

Train shape: torch.Size([1094, 30, 4]), torch.Size([1094, 4])
Valid shape: torch.Size([121, 30, 4]), torch.Size([121, 4])


## XGboost

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.ensemble import VotingClassifier
import joblib
import warnings
import os
import json  # Add json for saving parameters
import time
warnings.filterwarnings('ignore')

# VN30 symbols list
VN30 = [
    'ACB', 'BCM', 'BID', 'BVH', 'CTG',
    'FPT', 'GAS', 'GVR', 'HDB', 'HPG',
    'LPB', 'MBB', 'MSN', 'MWG', 'PLX',
    'SAB', 'SHB', 'SSB', 'SSI', 'STB',
    'TCB', 'TPB', 'VCB', 'VHM', 'VIB',
    'VIC', 'VJC', 'VNM', 'VPB', 'VRE',
]

# Create necessary directories
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('best_params', exist_ok=True)  # Add best_params folder
os.makedirs('data/vn30/multi_class_classification', exist_ok=True)
os.makedirs('results_summary', exist_ok=True)  # Add results summary folder

# Global variables for tracking results
all_results = []
processing_log = []

print("🚀 STARTING VN30 BATCH PROCESSING...")
print(f"📊 Total symbols to process: {len(VN30)}")
print("=" * 60)

# Define label mapping globally
label_mapping = {
    'strong_down': 0,
    'weak_down': 1,
    'sideways': 2,
    'weak_up': 3,
    'strong_up': 4
}
reverse_mapping = {v: k for k, v in label_mapping.items()}

Original shape: (1094, 30, 4)
Flattened shapes - X_train: (1094, 120), X_valid: (121, 120), X_test: (298, 120)
Class distribution - Train: [218 220 217 217 222]
Class distribution - Valid: [23 25 23 24 26]
Class distribution - Test: [56 59 60 62 61]


In [ ]:
# Feature Engineering: Add technical indicators
def preprocess_market_data(df):
    """Preprocess market data with focus on trends and price patterns"""
    # Copy data to avoid modifying original
    data = df.copy()

    # 1. Basic features: Price and volume volatility
    data['price_range'] = (data['high'] - data['low']) / data['close']  # Daily range
    data['close_to_open'] = (data['close'] - data['open']) / data['open']  # Intraday % change
    data['high_to_open'] = (data['high'] - data['open']) / data['open']  # Max % gain from open
    data['low_to_open'] = (data['low'] - data['open']) / data['open']  # Max % loss from open

    if 'volume' in data.columns:
        # Volume moving averages
        data['volume_ma5'] = data['volume'].rolling(window=5).mean()
        data['volume_ma10'] = data['volume'].rolling(window=10).mean()
        data['rel_volume'] = data['volume'] / data['volume_ma5']  # Relative volume

    # 2. Trend features
    # SMA - Simple Moving Average
    for window in [5, 10, 20, 50]:
        data[f'sma_{window}'] = data['close'].rolling(window=window).mean()
        data[f'close_to_sma_{window}'] = data['close'] / data[f'sma_{window}'] - 1

    # EMA - Exponential Moving Average
    for window in [5, 10, 20, 50]:
        data[f'ema_{window}'] = data['close'].ewm(span=window, adjust=False).mean()
        data[f'close_to_ema_{window}'] = data['close'] / data[f'ema_{window}'] - 1

    # 3. Momentum features
    # Percentage change over various periods
    for period in [1, 3, 5, 10, 20]:
        data[f'return_{period}d'] = data['close'].pct_change(periods=period)

    # RSI - Relative Strength Index
    for window in [6, 14]:
        delta = data['close'].diff()
        gain = delta.where(delta > 0, 0).rolling(window=window).mean()
        loss = -delta.where(delta < 0, 0).rolling(window=window).mean()

        rs = gain / loss
        data[f'rsi_{window}'] = 100 - (100 / (1 + rs))

    # MACD
    data['ema_12'] = data['close'].ewm(span=12, adjust=False).mean()
    data['ema_26'] = data['close'].ewm(span=26, adjust=False).mean()
    data['macd'] = data['ema_12'] - data['ema_26']
    data['macd_signal'] = data['macd'].ewm(span=9, adjust=False).mean()
    data['macd_hist'] = data['macd'] - data['macd_signal']

    # 4. Volatility features
    # Bollinger Bands
    for window in [20]:
        data[f'bb_middle_{window}'] = data['close'].rolling(window=window).mean()
        data[f'bb_std_{window}'] = data['close'].rolling(window=window).std()
        data[f'bb_upper_{window}'] = data[f'bb_middle_{window}'] + 2 * data[f'bb_std_{window}']
        data[f'bb_lower_{window}'] = data[f'bb_middle_{window}'] - 2 * data[f'bb_std_{window}']
        data[f'bb_width_{window}'] = (data[f'bb_upper_{window}'] - data[f'bb_lower_{window}']) / data[f'bb_middle_{window}']
        data[f'bb_position_{window}'] = (data['close'] - data[f'bb_lower_{window}']) / (data[f'bb_upper_{window}'] - data[f'bb_lower_{window}'])

    # ATR - Average True Range (volatility)
    data['tr'] = np.maximum(
        data['high'] - data['low'],
        np.maximum(
            abs(data['high'] - data['close'].shift(1)),
            abs(data['low'] - data['close'].shift(1))
        )
    )
    data['atr_14'] = data['tr'].rolling(window=14).mean()
    data['atr_ratio'] = data['atr_14'] / data['close']

    # 5. Candlestick pattern features
    # Body size
    data['body_size'] = abs(data['close'] - data['open']) / data['close']
    # Upper and lower shadows
    data['upper_shadow'] = (data['high'] - data[['open', 'close']].max(axis=1)) / data['close']
    data['lower_shadow'] = (data[['open', 'close']].min(axis=1) - data['low']) / data['close']
    # Body ratio
    data['body_ratio'] = data['body_size'] / (data['high'] - data['low'])

    # Special candlestick patterns
    # Doji
    data['is_doji'] = (abs(data['close'] - data['open']) / (data['high'] - data['low']) < 0.1).astype(int)
    # Bullish/Bearish
    data['is_bullish'] = (data['close'] > data['open']).astype(int)
    data['is_bearish'] = (data['close'] < data['open']).astype(int)
    # Hammer/Shooting Star
    data['is_hammer'] = ((data['lower_shadow'] > 2 * data['body_size']) &
                         (data['upper_shadow'] < 0.3 * data['body_size'])).astype(int)

    # 6. Recent price movement features
    # Consecutive up/down days
    data['streak'] = 0
    streak = 0
    for i in range(1, len(data)):
        if data['close'].iloc[i] > data['close'].iloc[i-1]:
            if streak <= 0:
                streak = 1
            else:
                streak += 1
        elif data['close'].iloc[i] < data['close'].iloc[i-1]:
            if streak >= 0:
                streak = -1
            else:
                streak -= 1
        else:
            streak = 0
        data['streak'].iloc[i] = streak

    # 7. Support/Resistance level features
    # Support and resistance levels (using 20-day high/low)
    data['support_20d'] = data['low'].rolling(window=20).min()
    data['resist_20d'] = data['high'].rolling(window=20).max()
    data['dist_to_support'] = (data['close'] - data['support_20d']) / data['close']
    data['dist_to_resist'] = (data['resist_20d'] - data['close']) / data['close']

    # 8. Mixed features
    # Stochastic Oscillator
    data['lowest_14'] = data['low'].rolling(window=14).min()
    data['highest_14'] = data['high'].rolling(window=14).max()
    data['%K'] = 100 * ((data['close'] - data['lowest_14']) / (data['highest_14'] - data['lowest_14']))
    data['%D'] = data['%K'].rolling(window=3).mean()

    # 9. Gap Up/Down
    data['gap_up'] = ((data['low'] > data['high'].shift(1)).astype(int))
    data['gap_down'] = ((data['high'] < data['low'].shift(1)).astype(int))

    # 10. Composite trend features
    data['trend_strength'] = abs(data['close_to_sma_50'])
    data['trend_direction'] = np.sign(data['close_to_sma_50'])
    data['momentum'] = data['close_to_ema_5'] * data['rsi_14'] / 50

    # Handle NaN values
    data = data.replace([np.inf, -np.inf], np.nan)

    # Use forward fill first, then back fill (updated method)
    data = data.ffill()
    data = data.bfill()

    # Replace remaining NaN with 0
    data = data.fillna(0)

    return data



(1094, 120)
(121, 120)
(298, 120)


In [ ]:
# split data
def create_train_valid_test_sets(symbol, val_ratio=0.2):
    # Read data from CSV files
    train_valid_path = f'data/vn30/multi_class_classification/{symbol}_train.csv'
    test_path = f'data/vn30/multi_class_classification/{symbol}_test.csv'

    try:
        # Read training and validation data
        df_train_valid = pd.read_csv(train_valid_path, parse_dates=['time'])
        df_test = pd.read_csv(test_path, parse_dates=['time'])
    except FileNotFoundError as e:
        print(f"Error: File not found - {e}")
        print("Please ensure all data files exist in the correct directory structure.")
        return None

    # Apply preprocessing
    train_valid_processed = preprocess_market_data(df_train_valid)
    test_processed = preprocess_market_data(df_test)

    # Split validation set from training data
    val_size = int(len(train_valid_processed) * val_ratio)
    train_data = train_valid_processed.iloc[:-val_size].copy()
    valid_data = train_valid_processed.iloc[-val_size:].copy()
    test_data = test_processed.copy()

    # Convert labels to numbers
    for df in [train_data, valid_data, test_data]:
        df['label_num'] = df['label'].map(label_mapping)

    # Remove unnecessary columns
    drop_cols = ['time', 'label']
    X_train = train_data.drop(drop_cols + ['label_num'], axis=1)
    y_train = train_data['label_num']

    X_valid = valid_data.drop(drop_cols + ['label_num'], axis=1)
    y_valid = valid_data['label_num']

    X_test = test_data.drop(drop_cols + ['label_num'], axis=1)
    y_test = test_data['label_num']

    # Note: Tree-based models (LightGBM, XGBoost, CatBoost) don't require feature scaling
    # So we'll use the original data without StandardScaler for simplicity and consistency

    print(f"Data shapes - Train: {X_train.shape}, Valid: {X_valid.shape}, Test: {X_test.shape}")
    print(f"Label distribution - Train: {np.bincount(y_train)}")
    print(f"Label distribution - Valid: {np.bincount(y_valid)}")
    print(f"Label distribution - Test: {np.bincount(y_test)}")

    return X_train, y_train, X_valid, y_valid, X_test, y_test, X_train.columns


^C


In [ ]:
# Function to process a single symbol
def process_symbol(symbol):
    """Process a single symbol and return results"""
    print(f"\n{'='*50}")
    print(f"🔄 PROCESSING: {symbol}")
    print(f"{'='*50}")

    start_time = time.time()
    symbol_results = {
        'symbol': symbol,
        'status': 'failed',
        'error': None,
        'processing_time': 0,
        'models_performance': {},
        'best_iterations': {},
        'files_created': []
    }

    try:
        # Create training, validation and test data
        result = create_train_valid_test_sets(symbol)
        if result is None:
            symbol_results['error'] = 'Failed to load data files'
            return symbol_results

        X_train, y_train, X_valid, y_valid, X_test, y_test, feature_names = result

        print(f"✅ Data loaded successfully for {symbol}")

        # Create and train models
        print(f"🤖 Training models for {symbol}...")

        # LightGBM
        print(f"  📊 Training LightGBM...")
        lgb_model = lgb.LGBMClassifier(
            objective='multiclass',
            num_class=5,
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=0.1,
            min_child_samples=20,
            random_state=42,
            class_weight='balanced',
            verbose=-1
        )
        lgb_model.fit(
            X_train, y_train,
            eval_set=[(X_valid, y_valid)],
            eval_metric='multi_logloss',
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
        )

        # XGBoost
        print(f"  🌲 Training XGBoost...")
        xgb_model = XGBClassifier(
            objective='multi:softmax',
            num_class=5,
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=0.1,
            min_child_weight=3,
            gamma=0.1,
            random_state=42,
            verbosity=0,
            eval_metric='mlogloss',
            early_stopping_rounds=50
        )
        xgb_model.fit(
            X_train, y_train,
            eval_set=[(X_valid, y_valid)],
            verbose=False
        )

        # CatBoost
        print(f"  🐱 Training CatBoost...")
        cat_model = CatBoostClassifier(
            iterations=300,
            learning_rate=0.05,
            depth=6,
            l2_leaf_reg=3,
            random_strength=0.1,
            random_seed=42,
            verbose=0
        )
        cat_model.fit(
            X_train, y_train,
            eval_set=(X_valid, y_valid),
            early_stopping_rounds=50,
            verbose=False
        )

        # Save optimal parameters for all models
        print(f"💾 Saving parameters for {symbol}...")

        # Extract optimal parameters and performance metrics
        best_params = {
            'symbol': symbol,
            'timestamp': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
            'data_info': {
                'train_samples': len(X_train),
                'valid_samples': len(X_valid),
                'test_samples': len(X_test),
                'num_features': X_train.shape[1]
            },
            'models': {
                'lgb': {
                    'best_iteration': lgb_model.best_iteration if hasattr(lgb_model, 'best_iteration') else 'N/A',
                    'params': lgb_model.get_params(),
                    'feature_importance_top10': {}
                },
                'xgb': {
                    'best_iteration': getattr(xgb_model, 'best_iteration', 'N/A'),
                    'params': xgb_model.get_params(),
                    'feature_importance_top10': {}
                },
                'catboost': {
                    'best_iteration': getattr(cat_model, 'best_iteration_', 'N/A'),
                    'params': cat_model.get_params(),
                    'feature_importance_top10': {}
                }
            }
        }

        # Add feature importance (top 10) for each model
        for model_name, model in [('lgb', lgb_model), ('xgb', xgb_model), ('catboost', cat_model)]:
            try:
                if hasattr(model, 'feature_importances_'):
                    importance = model.feature_importances_
                    top_indices = np.argsort(importance)[-10:][::-1]
                    best_params['models'][model_name]['feature_importance_top10'] = {
                        feature_names[i]: float(importance[i]) for i in top_indices
                    }
            except Exception as e:
                print(f"Warning: Could not extract {model_name} feature importance: {e}")

        # Save to JSON file
        params_file = f'best_params/{symbol}_best_params.json'
        with open(params_file, 'w', encoding='utf-8') as f:
            json.dump(best_params, f, indent=2, ensure_ascii=False)

        # Also save a simplified version for quick reference
        optimal_iterations = {
            'lgb_best_iteration': best_params['models']['lgb']['best_iteration'],
            'xgb_best_iteration': best_params['models']['xgb']['best_iteration'],
            'catboost_best_iteration': best_params['models']['catboost']['best_iteration']
        }
        joblib.dump(optimal_iterations, f'best_params/{symbol}_optimal_iterations.pkl')

        # Create ensemble with optimal iterations
        print(f"🔗 Creating ensemble for {symbol}...")

        lgb_optimal_iter = best_params['models']['lgb']['best_iteration'] if best_params['models']['lgb']['best_iteration'] != 'N/A' else 200
        xgb_optimal_iter = best_params['models']['xgb']['best_iteration'] if best_params['models']['xgb']['best_iteration'] != 'N/A' else 200
        cat_optimal_iter = best_params['models']['catboost']['best_iteration'] if best_params['models']['catboost']['best_iteration'] != 'N/A' else 200

        # Create ensemble models
        lgb_ensemble = lgb.LGBMClassifier(
            objective='multiclass', num_class=5, n_estimators=int(lgb_optimal_iter),
            learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, min_child_samples=20,
            random_state=42, class_weight='balanced', verbose=-1
        )

        xgb_ensemble = XGBClassifier(
            objective='multi:softmax', num_class=5, n_estimators=int(xgb_optimal_iter),
            learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, min_child_weight=3, gamma=0.1,
            random_state=42, verbosity=0
        )

        cat_ensemble = CatBoostClassifier(
            iterations=int(cat_optimal_iter), learning_rate=0.05, depth=6,
            l2_leaf_reg=3, random_strength=0.1, random_seed=42, verbose=0
        )

        ensemble = VotingClassifier(
            estimators=[('lgb', lgb_ensemble), ('xgb', xgb_ensemble), ('cat', cat_ensemble)],
            voting='soft'
        )
        ensemble.fit(X_train, y_train)

        # Quick evaluation (without plots for batch processing)
        print(f"📊 Evaluating models for {symbol}...")

        def quick_evaluate(model, X_test, y_test, model_name):
            y_pred = model.predict(X_test)
            y_pred = np.array(y_pred).flatten().astype(int)
            acc = balanced_accuracy_score(y_test, y_pred)
            return acc

        # Get performance metrics
        lgb_acc = quick_evaluate(lgb_model, X_test, y_test, "LightGBM")
        xgb_acc = quick_evaluate(xgb_model, X_test, y_test, "XGBoost")
        cat_acc = quick_evaluate(cat_model, X_test, y_test, "CatBoost")
        ens_acc = quick_evaluate(ensemble, X_test, y_test, "Ensemble")

        # Save ensemble model
        model_path = f'checkpoints/ensemble_classifier_{symbol}.pkl'
        joblib.dump(ensemble, model_path)

        # Update results
        symbol_results.update({
            'status': 'success',
            'processing_time': time.time() - start_time,
            'models_performance': {
                'lgb_accuracy': lgb_acc,
                'xgb_accuracy': xgb_acc,
                'cat_accuracy': cat_acc,
                'ensemble_accuracy': ens_acc
            },
            'best_iterations': optimal_iterations,
            'files_created': [params_file, f'best_params/{symbol}_optimal_iterations.pkl', model_path]
        })

        print(f"✅ {symbol} completed successfully!")
        print(f"   💯 Accuracies - LGB: {lgb_acc:.4f}, XGB: {xgb_acc:.4f}, CAT: {cat_acc:.4f}, ENS: {ens_acc:.4f}")
        print(f"   ⏱️  Processing time: {symbol_results['processing_time']:.1f}s")

    except Exception as e:
        symbol_results['error'] = str(e)
        print(f"❌ Error processing {symbol}: {e}")

    return symbol_results


In [ ]:
# main
print("\n🚀 STARTING BATCH PROCESSING OF ALL VN30 SYMBOLS...")
total_start_time = time.time()

for i, symbol in enumerate(VN30, 1):
    print(f"\n📍 Progress: {i}/{len(VN30)} symbols")

    # Process single symbol
    result = process_symbol(symbol)
    all_results.append(result)

    # Log processing status
    status_emoji = "✅" if result['status'] == 'success' else "❌"
    processing_log.append(f"{status_emoji} {symbol}: {result['status']}")

    # Brief progress update
    if result['status'] == 'success':
        best_acc = max(result['models_performance'].values())
        print(f"   🎯 Best accuracy: {best_acc:.4f}")

# Generate comprehensive summary
print("\n" + "="*80)
print("🎯 BATCH PROCESSING COMPLETED!")
print("="*80)

total_time = time.time() - total_start_time
successful_symbols = [r for r in all_results if r['status'] == 'success']
failed_symbols = [r for r in all_results if r['status'] == 'failed']

print(f"\n📊 OVERALL STATISTICS:")
print(f"   ✅ Successful: {len(successful_symbols)}/{len(VN30)} symbols")
print(f"   ❌ Failed: {len(failed_symbols)}/{len(VN30)} symbols")
print(f"   ⏱️  Total time: {total_time:.1f}s ({total_time/60:.1f} minutes)")
print(f"   📈 Average time per symbol: {total_time/len(VN30):.1f}s")

if successful_symbols:
    print(f"\n🏆 PERFORMANCE SUMMARY:")

    # Collect all accuracies
    all_accuracies = {
        'lgb': [r['models_performance']['lgb_accuracy'] for r in successful_symbols],
        'xgb': [r['models_performance']['xgb_accuracy'] for r in successful_symbols],
        'cat': [r['models_performance']['cat_accuracy'] for r in successful_symbols],
        'ensemble': [r['models_performance']['ensemble_accuracy'] for r in successful_symbols]
    }

    for model_name, accuracies in all_accuracies.items():
        avg_acc = np.mean(accuracies)
        std_acc = np.std(accuracies)
        min_acc = np.min(accuracies)
        max_acc = np.max(accuracies)
        print(f"   {model_name.upper():8}: {avg_acc:.4f} ± {std_acc:.4f} (min: {min_acc:.4f}, max: {max_acc:.4f})")

    # Best performing symbols
    print(f"\n🥇 TOP 5 PERFORMERS (by ensemble accuracy):")
    top_performers = sorted(successful_symbols,
                          key=lambda x: x['models_performance']['ensemble_accuracy'],
                          reverse=True)[:5]

    for i, result in enumerate(top_performers, 1):
        symbol = result['symbol']
        acc = result['models_performance']['ensemble_accuracy']
        print(f"   {i}. {symbol}: {acc:.4f}")

if failed_symbols:
    print(f"\n❌ FAILED SYMBOLS:")
    for result in failed_symbols:
        print(f"   {result['symbol']}: {result['error']}")

# Save comprehensive results
results_summary = {
    'processing_info': {
        'total_symbols': len(VN30),
        'successful_symbols': len(successful_symbols),
        'failed_symbols': len(failed_symbols),
        'total_processing_time': total_time,
        'timestamp': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
    },
    'performance_summary': {},
    'individual_results': all_results
}

if successful_symbols:
    results_summary['performance_summary'] = {
        model_name: {
            'mean': float(np.mean(accuracies)),
            'std': float(np.std(accuracies)),
            'min': float(np.min(accuracies)),
            'max': float(np.max(accuracies))
        }
        for model_name, accuracies in all_accuracies.items()
    }

# Save results
summary_file = f'results_summary/vn30_batch_results_{pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")}.json'
with open(summary_file, 'w', encoding='utf-8') as f:
    json.dump(results_summary, f, indent=2, ensure_ascii=False)

print(f"\n💾 RESULTS SAVED:")
print(f"   📋 Summary: {summary_file}")
print(f"   📁 Models: checkpoints/ folder ({len(successful_symbols)} files)")
print(f"   ⚙️  Parameters: best_params/ folder ({len(successful_symbols)} x 2 files)")

print(f"\n🎉 BATCH PROCESSING COMPLETE!")
print("="*80)
